# Checkpoints


In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

source_str = "orders"
sink_str = "orders_aggregated"

tn = (
    Tn.source(source_str)
    .map(lambda r: r["value"])
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                    "product_ids": sorted(agg_r["product_ids"] + [r["product_id"]])},
                  {"orders": 0, "product_ids": []},
                  lambda by, agg_r: {"customer_id": by,
                                     "orders": agg_r["orders"],
                                     "product_ids": agg_r["product_ids"]})
    .map(lambda r: {"key": r["customer_id"],
                    "value": r})
    .sink(sink_str)
)

built_tn = Tn.build(tn)



In [1]:
import random

class OrderGenerator:
    def __init__(self):
        self.order_id_int = 0
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = 1

    def generate(self):
        m = {
            "key": self.order_id_int,
            "value": {"id": self.order_id_int,
                      "product_id": random.randint(0, 100 - 1),
                      "customer_id": random.randint(0, 10 - 1),
                      "ts": self.ts_int},
        }
        #
        self.order_id_int += 1
        #
        self.ts_int += self.ts_step_int
        #
        return m

#

gen = OrderGenerator()
for _ in range(3):
    print(gen.generate())


{'key': 0, 'value': {'id': 0, 'product_id': 34, 'customer_id': 1, 'ts': 0}}
{'key': 1, 'value': {'id': 1, 'product_id': 78, 'customer_id': 4, 'ts': 1}}
{'key': 2, 'value': {'id': 2, 'product_id': 69, 'customer_id': 7, 'ts': 2}}


In [ ]:
built_tn.reset()
gen = OrderGenerator()
source_m_list = []
sink_m_list = []
for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

source_key_int_value_dict_dict = {}
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["key"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


In [ ]:
import cloudpickle

#

gen = OrderGenerator()


#

built_tn.reset()
source_m_list = []
sink_m_list = []
for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

x = cloudpickle.dumps(built_tn._evaluator)

# print(built_tn.latest())

built_tn.reset()

# print(built_tn.latest())

y = cloudpickle.loads(x)

built_tn._evaluator = y

# print(built_tn.latest())

#

for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

source_key_int_value_dict_dict = {}
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["key"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


In [10]:
import sys
sys.path.insert(1, "..")

import kafi.streams.streams
import importlib
importlib.reload(kafi.streams.streams)

from kafi.kafka.cluster.cluster import Cluster
from kafi.streams.streams import Streams

import logging
logging.basicConfig(level=logging.DEBUG)

c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}})

source_str = "orders"
sink_str = "orders_aggregated"

tn = (
    Streams.source(c, source_str)
    
    .map(lambda r: r["value"])
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                    "order_ids": sorted(agg_r["order_ids"] + [r["id"]]),
                                    "product_ids": sorted(agg_r["product_ids"] + [r["product_id"]])},
                  {"orders": 0, "order_ids": [], "product_ids": []},
                  lambda by, agg_r: {"customer_id": by,
                                     "orders": agg_r["orders"],
                                     "order_ids": agg_r["order_ids"],
                                     "product_ids": agg_r["product_ids"]})
    .map(lambda r: {"key": r["customer_id"],
                    "value": r}).peek("sink")
    .sink(c, sink_str)
)

built_tn = Streams.build(tn)



In [ ]:
from kafi.helpers import get_millis

orders_int = 1000

built_tn.reset()

checkpoint_str = "checkpoint"
g = f"group_{get_millis()}"

c.recreate(source_str)
c.recreate(sink_str)
c.recreate(checkpoint_str)

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, checkpoint_interval=0.01, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)
gen = OrderGenerator()

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



'orders'

DEBUG:kafi.streams.streams:Checkpoint consumer group ('group_1785154943508_checkpoint') offsets for topic 'checkpoint': {}


(['checkpoint'], 'group_1785154943508_checkpoint')


Reading: 0 msg [00:05, ? msg/s]
DEBUG:kafi.streams.streams:Source consumer group ('group_1785154943508') offsets for topic 'orders': {}


(['orders'], 'group_1785154943508')


INFO:kafi.streams.streams:Saving checkpoint...
INFO:kafi.streams.streams:...saving checkpoint done (29 KB compressed, 195 uncompressed).
INFO:kafi.streams.streams:Committed {0: 1000} for source orders.


sink: {'key': 2, 'value': {'customer_id': 2, 'orders': 116, 'order_ids': [0, 1, 27, 28, 57, 72, 85, 97, 98, 109, 110, 111, 121, 127, 135, 148, 152, 153, 172, 174, 182, 187, 188, 214, 228, 238, 239, 253, 261, 274, 285, 294, 296, 313, 314, 316, 330, 331, 337, 341, 353, 374, 376, 380, 383, 387, 406, 410, 412, 420, 429, 430, 434, 435, 439, 463, 468, 477, 483, 499, 507, 513, 515, 542, 550, 559, 566, 593, 597, 598, 609, 614, 626, 637, 643, 649, 654, 655, 656, 659, 669, 679, 687, 696, 698, 725, 733, 735, 736, 767, 768, 776, 786, 789, 794, 799, 831, 851, 855, 857, 859, 862, 870, 871, 873, 881, 898, 917, 932, 941, 943, 952, 958, 964, 972, 975], 'product_ids': [2, 2, 2, 4, 4, 4, 5, 5, 5, 7, 7, 9, 9, 12, 12, 12, 13, 13, 13, 15, 17, 19, 19, 20, 22, 22, 22, 24, 25, 25, 28, 28, 28, 29, 31, 33, 33, 34, 34, 35, 36, 37, 37, 37, 39, 39, 40, 40, 40, 41, 41, 41, 43, 43, 44, 44, 46, 46, 46, 46, 48, 49, 50, 52, 52, 53, 53, 54, 55, 57, 58, 58, 58, 61, 61, 61, 62, 62, 62, 65, 65, 65, 67, 68, 68, 69, 70, 72, 7

In [12]:
await stop_fun()
await Streams.tasks()

INFO:kafi.streams.streams:Safely stopping Streams...
INFO:kafi.streams.streams:...done.


[]

In [ ]:
built_tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



{'orders': 1000}
{'orders_aggregated': 10}
{'checkpoint': 31}


'orders'

DEBUG:kafi.streams.streams:Checkpoint consumer group ('group_1785154943508_checkpoint') offsets for topic 'checkpoint': {}


(['checkpoint'], 'group_1785154943508_checkpoint')


Reading: 1 msg [00:05,  5.21s/ msg]
INFO:kafi.streams.streams:Loading checkpoint...
INFO:kafi.streams.streams:...loading checkpoint done (29 KB compressed, 195 uncompressed).
DEBUG:kafi.streams.streams:Source consumer group ('group_1785154943508') offsets for topic 'orders': {0: 1000}
DEBUG:kafi.streams.streams:Source consumer group offsets for topic 'orders' overridden by checkpoint offsets: {0: 1000}


(['orders'], 'group_1785154943508')


INFO:kafi.streams.streams:Saving checkpoint...
INFO:kafi.streams.streams:...saving checkpoint done (55 KB compressed, 413 uncompressed).
INFO:kafi.streams.streams:Committed {0: 2000} for source orders.


sink: {'key': 3, 'value': {'customer_id': 3, 'orders': 195, 'order_ids': [11, 37, 46, 52, 54, 62, 114, 130, 132, 140, 143, 151, 154, 160, 161, 175, 177, 179, 183, 186, 196, 204, 219, 229, 241, 251, 272, 273, 283, 291, 300, 334, 340, 348, 375, 397, 409, 418, 432, 438, 440, 456, 459, 478, 491, 512, 518, 536, 548, 561, 568, 570, 573, 587, 595, 661, 667, 670, 673, 674, 690, 701, 720, 726, 728, 730, 754, 756, 758, 762, 764, 769, 798, 800, 808, 826, 832, 860, 869, 878, 888, 901, 904, 908, 912, 920, 928, 933, 944, 948, 959, 974, 977, 985, 991, 992, 993, 1000, 1014, 1015, 1016, 1021, 1042, 1043, 1056, 1073, 1084, 1092, 1112, 1113, 1114, 1149, 1160, 1161, 1169, 1173, 1184, 1234, 1245, 1249, 1317, 1321, 1324, 1338, 1347, 1355, 1362, 1363, 1378, 1399, 1407, 1434, 1450, 1463, 1494, 1495, 1496, 1505, 1550, 1562, 1572, 1577, 1591, 1592, 1601, 1603, 1604, 1608, 1610, 1628, 1661, 1664, 1666, 1675, 1680, 1690, 1696, 1702, 1716, 1721, 1742, 1747, 1752, 1762, 1770, 1772, 1788, 1791, 1792, 1801, 1806, 180

In [14]:
await stop_fun()
await Streams.tasks()

INFO:kafi.streams.streams:Safely stopping Streams...
INFO:kafi.streams.streams:...done.


[]

In [ ]:
built_tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()


{'orders': 2000}
{'orders_aggregated': 20}
{'checkpoint': 88}


'orders'

DEBUG:kafi.streams.streams:Checkpoint consumer group ('group_1785154943508_checkpoint') offsets for topic 'checkpoint': {0: 31}


(['checkpoint'], 'group_1785154943508_checkpoint')


Reading: 1 msg [00:05,  5.22s/ msg]
INFO:kafi.streams.streams:Loading checkpoint...
INFO:kafi.streams.streams:...loading checkpoint done (55 KB compressed, 413 uncompressed).
DEBUG:kafi.streams.streams:Source consumer group ('group_1785154943508') offsets for topic 'orders': {0: 2000}
DEBUG:kafi.streams.streams:Source consumer group offsets for topic 'orders' overridden by checkpoint offsets: {0: 2000}


(['orders'], 'group_1785154943508')


INFO:kafi.streams.streams:Saving checkpoint...
INFO:kafi.streams.streams:...saving checkpoint done (72 KB compressed, 517 uncompressed).
INFO:kafi.streams.streams:Committed {0: 3000} for source orders.


sink: {'key': 5, 'value': {'customer_id': 5, 'orders': 286, 'order_ids': [5, 23, 30, 38, 51, 65, 68, 76, 81, 101, 117, 133, 137, 141, 145, 169, 195, 203, 222, 232, 236, 240, 244, 260, 269, 275, 288, 293, 297, 308, 320, 322, 345, 357, 382, 423, 447, 448, 452, 460, 466, 467, 472, 473, 475, 482, 488, 494, 497, 498, 510, 521, 532, 543, 546, 569, 584, 601, 616, 630, 631, 652, 689, 692, 731, 734, 742, 743, 745, 747, 751, 753, 761, 818, 828, 842, 856, 863, 875, 882, 891, 900, 903, 905, 923, 930, 938, 945, 956, 960, 995, 1006, 1011, 1027, 1030, 1039, 1044, 1068, 1069, 1078, 1096, 1108, 1109, 1119, 1120, 1122, 1131, 1132, 1156, 1171, 1178, 1197, 1205, 1223, 1228, 1231, 1247, 1278, 1282, 1289, 1304, 1306, 1312, 1318, 1335, 1336, 1348, 1359, 1365, 1376, 1395, 1411, 1425, 1427, 1438, 1439, 1440, 1443, 1461, 1464, 1472, 1482, 1483, 1517, 1531, 1542, 1543, 1551, 1560, 1578, 1585, 1593, 1598, 1611, 1625, 1630, 1632, 1640, 1646, 1652, 1655, 1663, 1674, 1688, 1695, 1720, 1732, 1748, 1768, 1776, 1800, 1

In [16]:
await stop_fun()
await Streams.tasks()

INFO:kafi.streams.streams:Safely stopping Streams...
INFO:kafi.streams.streams:...done.


[]

In [17]:
source_key_int_value_dict_dict = {}
source_m_list = c.cat(source_str)
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_order_id_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("order_ids", [])
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "order_ids": sorted(agg_order_id_int_list + [order_id_int]),
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
sink_m_list = c.cat(sink_str)
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["value"]["customer_id"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


(['orders'], '1785154980972')


Reading: 3000 msg [00:05, 573.98 msg/s]


(['orders_aggregated'], '1785154986236')


Reading: 30 msg [00:05,  5.75 msg/s]

{2: {'customer_id': 2, 'orders': 304, 'order_ids': [0, 1, 27, 28, 57, 72, 85, 97, 98, 109, 110, 111, 121, 127, 135, 148, 152, 153, 172, 174, 182, 187, 188, 214, 228, 238, 239, 253, 261, 274, 285, 294, 296, 313, 314, 316, 330, 331, 337, 341, 353, 374, 376, 380, 383, 387, 406, 410, 412, 420, 429, 430, 434, 435, 439, 463, 468, 477, 483, 499, 507, 513, 515, 542, 550, 559, 566, 593, 597, 598, 609, 614, 626, 637, 643, 649, 654, 655, 656, 659, 669, 679, 687, 696, 698, 725, 733, 735, 736, 767, 768, 776, 786, 789, 794, 799, 831, 851, 855, 857, 859, 862, 870, 871, 873, 881, 898, 917, 932, 941, 943, 952, 958, 964, 972, 975, 1003, 1013, 1017, 1024, 1034, 1036, 1055, 1075, 1077, 1081, 1087, 1102, 1103, 1126, 1137, 1143, 1147, 1154, 1155, 1165, 1170, 1190, 1199, 1203, 1208, 1210, 1212, 1218, 1232, 1235, 1237, 1246, 1255, 1277, 1279, 1293, 1302, 1341, 1344, 1352, 1356, 1385, 1386, 1388, 1390, 1392, 1397, 1410, 1414, 1415, 1416, 1419, 1421, 1424, 1466, 1502, 1513, 1526, 1527, 1535, 1540, 1565, 1567, 1